In [7]:
!pip install transformers --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 47.6 MB/s eta 0:00:00


In [1]:
!pip install torch torchaudio

In [13]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch
import torchaudio

In [14]:
model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [15]:
import requests

url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"

r = requests.get(url)

with open("sample.flac", "wb") as f:
    f.write(r.content)

audio, sampling_rate = torchaudio.load("/content/sample.flac")


In [16]:
if sampling_rate != 16000:
    resampler = torchaudio.transforms.Resample(sampling_rate, 16000)
    audio = resampler(audio)

In [17]:
inputs = processor(
    audio.squeeze().numpy(),
    sampling_rate=16000,
    return_tensors="pt"
)

In [18]:
with torch.no_grad():
    predicted_ids = model.generate(inputs["input_features"])

In [19]:
transcription = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0]

print("Transcription:", transcription)

Transcription:  He hoped there would be stew for dinner, turnips and carrots and bruised potatoes and fat mutton pieces to be ladled out in thick, peppered, flour-fattened sauce.
